1. Import Libraries

## 1. Import Libraries

In [228]:
import pandas as pd
import numpy as np
import re

## 2. Baca Dataset sleep health yang kotor

In [229]:
# Hubungkan Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Lokasi file di Drive
file_path = '/content/drive/MyDrive/Colab Notebooks/DATMIN/sleep_health_data_kotor.csv'

# Baca file CSV
df = pd.read_csv(file_path, dtype=str, keep_default_na=False)

# Tampilkan 10 baris pertama
df.head(10)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,No,Age,Gender,Sleep Duration (hrs),sleep_quality_score,Exercise Day,STEPS_THAT_DAY,stress score,heart_rate_resting_bpm,screen_time_before_bed_mins,Caffeine_mg,alcohol units,nap_duration_mins
0,23,27,Female,7.78,7.4,1,5587,2.5,63,1.5 jam,0,0,69
1,28,34,Female,5.36,15,1,12520,7.9,57,38,0,0,0
2,35,57,Female,5.88,4.1,1,10650,99,63,27,0,0,0
3,16,24,Female,6.95,4.9,1,13465,4.1,56,110,0,-2,76
4,19,31,Female,delapan,4.9,1,10616,5.4,75,83,0,0,0
5,30,19,,3.77,2.6,0,8207,7.3,63,35,0,0,33
6,33,20,Female,7.99,6.3,TRUE,12409,4.3,59,24,0,0,0
7,31,dua puluh,Female,4.7,2.3,1,7463,7.5,70,101,300,2,0
8,11,18,Male,4.97,2.8,1,8524,8.8,67,53,0,0,0
9,2,55,Female,8.32,6.9,1,10111,4,52,33,0,0,8


## 3. Menampilkan struktur variabel dan kondisi data

In [230]:
# Sel kosong terbaca sebagai teks kosong, ubah jadi NaN agar bisa dihitung
df = df.replace(r'^\s*$', np.nan, regex=True)

print('Jumlah baris, kolom:', df.shape)
df.info()

Jumlah baris, kolom: (54, 13)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 54 entries, 0 to 53
Data columns (total 13 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   No                           54 non-null     object
 1   Age                          53 non-null     object
 2   Gender                       53 non-null     object
 3   Sleep Duration (hrs)         53 non-null     object
 4   sleep_quality_score          53 non-null     object
 5   Exercise Day                 54 non-null     object
 6   STEPS_THAT_DAY               53 non-null     object
 7   stress score                 53 non-null     object
 8   heart_rate_resting_bpm       53 non-null     object
 9   screen_time_before_bed_mins  54 non-null     object
 10  Caffeine_mg                  53 non-null     object
 11  alcohol units                54 non-null     object
 12  nap_duration_mins            53 non-null     object
dtypes: obje

In [231]:
print('Missing value per kolom:')
print(df.isna().sum())
print()
print('Baris duplikat:', df.duplicated().sum())

Missing value per kolom:
No                             0
Age                            1
Gender                         1
Sleep Duration (hrs)           1
sleep_quality_score            1
Exercise Day                   0
STEPS_THAT_DAY                 1
stress score                   1
heart_rate_resting_bpm         1
screen_time_before_bed_mins    0
Caffeine_mg                    1
alcohol units                  0
nap_duration_mins              1
dtype: int64

Baris duplikat: 4


## 4. Data Cleansing-Standarisasi nama kolom

In [232]:
# Pemetaan nama kolom lama -> nama kolom baku
rename_map = {
    'No': 'no',
    'Age': 'age',
    'Gender': 'gender',
    'Sleep Duration (hrs)': 'sleep_duration_hrs',
    'sleep_quality_score': 'sleep_quality_score',
    'Exercise Day': 'exercise_day',
    'STEPS_THAT_DAY': 'steps_that_day',
    'stress score': 'stress_score',
    'heart_rate_resting_bpm ': 'heart_rate_resting_bpm',
    'screen_time_before_bed_mins': 'screen_time_before_bed_mins',
    'Caffeine_mg': 'caffeine_mg_before_bed',
    'alcohol units': 'alcohol_units_before_bed',
    'nap_duration_mins': 'nap_duration_mins',
}

# Terapkan penggantian nama
df = df.rename(columns=rename_map)

# Cek hasilnya
print(df.columns.tolist())

['no', 'age', 'gender', 'sleep_duration_hrs', 'sleep_quality_score', 'exercise_day', 'steps_that_day', 'stress_score', 'heart_rate_resting_bpm', 'screen_time_before_bed_mins', 'caffeine_mg_before_bed', 'alcohol_units_before_bed', 'nap_duration_mins']


## 5. Data Cleansing-Standarisasi kolom No (Primary Key)

In [233]:
# Pastikan kolom no berupa angka
df['no'] = pd.to_numeric(df['no'], errors='coerce')

# Cek kondisi kolom no
print('Nomor kosong/tidak valid :', df['no'].isna().sum())
print('Rentang nomor            :', df['no'].min(), 'sampai', df['no'].max())
print('Jumlah nomor unik        :', df['no'].nunique(), 'dari', len(df))

# Tampilkan nomor yang muncul lebih dari sekali
df[df['no'].duplicated(keep=False)].sort_values('no')

Nomor kosong/tidak valid : 0
Rentang nomor            : 1 sampai 50
Jumlah nomor unik        : 50 dari 54


,no,age,gender,sleep_duration_hrs,sleep_quality_score,exercise_day,steps_that_day,stress_score,heart_rate_resting_bpm,screen_time_before_bed_mins,caffeine_mg_before_bed,alcohol_units_before_bed,nap_duration_mins
1,28,34,Female,5.36,15,1,12520,7.9,57,38,0,0,0
14,28,34,Female,5.36,15,1,12520,7.9,57,38,0,0,0
23,36,45,Wanita,6.44,5.2,1,6350,5.1,78,58,80,0,0
42,36,45,Wanita,6.44,5.2,1,6350,5.1,78,58,80,0,0
39,39,40,Female,7.18,4.6,0,2704,4.7,66,34,0,2,0
38,39,40,Female,7.18,4.6,0,2704,4.7,66,34,0,2,0
12,41,22,Male,4.05,2.3,0,4477,5.8,64,133,0,2,17
37,41,22,Male,4.05,2.3,0,4477,5.8,64,133,0,2,17


## 6. Data Cleansing-Standarisasi kolom Gender dan Exercise Day

In [234]:
print('Nilai gender sebelum       :', sorted(df['gender'].dropna().unique()))
print('Nilai exercise_day sebelum :', sorted(df['exercise_day'].dropna().unique()))

Nilai gender sebelum       : [' Male', 'F', 'FEMALE', 'Female', 'Female ', 'M', 'Male', 'Pria', 'Wanita', 'male']
Nilai exercise_day sebelum : ['0', '1', 'TRUE', 'Ya', 'Yes', 'no']


##Buat fungsi Pembersih
* Mengecek data kosong
*  Membersihkan spasi dan huruf
*  Mengubah nilai sesuai pemetaan





In [235]:
# Pemetaan: penulisan nilai baku
gender_map = {
    'male': 'Male', 'm': 'Male', 'pria': 'Male', 'laki-laki': 'Male',
    'female': 'Female', 'f': 'Female', 'wanita': 'Female', 'perempuan': 'Female',
}
bool_map = {
    '1': 1, 'yes': 1, 'ya': 1, 'true': 1, 'y': 1,
    '0': 0, 'no': 0, 'tidak': 0, 'false': 0, 'n': 0,
}

# Fungsi pembersih teks
def bersihkan_teks(nilai, pemetaan):
    if pd.isna(nilai):
        return np.nan
    nilai = re.sub(r'\s+', ' ', str(nilai)).strip().lower()
    return pemetaan.get(nilai, np.nan)

#Terapkankan ke kolom dan bandingkan

In [236]:
# Terapkan fungsi pada kolom (hasil ke kolom baru dulu, supaya bisa dibandingkan)
df['gender_bersih'] = df['gender'].apply(lambda x: bersihkan_teks(x, gender_map))
df['exercise_day_bersih'] = df['exercise_day'].apply(lambda x: bersihkan_teks(x, bool_map))

# Tampilkan sebelum dan sesudah
df[['gender', 'gender_bersih', 'exercise_day', 'exercise_day_bersih']].head(15)

,gender,gender_bersih,exercise_day,exercise_day_bersih
0,Female,Female,1,1
1,Female,Female,1,1
2,Female,Female,1,1
3,Female,Female,1,1
4,Female,Female,1,1
5,NaN,NaN,0,0
6,Female,Female,TRUE,1
7,Female,Female,1,1
8,Male,Male,1,1
9,Female,Female,1,1


#Pakai kolom bersih sebagai kolom utama

In [237]:
# Ganti kolom lama dengan kolom bersih, lalu buang kolom sementara
df['gender'] = df['gender_bersih']
df['exercise_day'] = df['exercise_day_bersih']
df = df.drop(columns=['gender_bersih', 'exercise_day_bersih'])


print('Nilai gender sesudah       :', df['gender'].dropna().unique().tolist())
print('Nilai exercise_day sesudah :', df['exercise_day'].dropna().unique().tolist())
print()
print('Gender kosong       :', df['gender'].isna().sum())
print('Exercise_day kosong :', df['exercise_day'].isna().sum())

Nilai gender sesudah       : ['Female', 'Male']
Nilai exercise_day sesudah : [1, 0]

Gender kosong       : 1
Exercise_day kosong : 0


In [238]:
print(bersihkan_teks(' Male', gender_map))
print(bersihkan_teks('FEMALE', gender_map))
print(bersihkan_teks('Wanita', gender_map))
print(bersihkan_teks('Yes', bool_map))

Male
Female
Female
1


# 7. Data Cleansing-Standarisasi kolom *numerik*

In [239]:
# Contoh nilai yang formatnya tidak standar (bukan angka murni)
kolom_cek = ['age', 'sleep_duration_hrs', 'steps_that_day', 'stress_score',
             'heart_rate_resting_bpm', 'screen_time_before_bed_mins',
             'caffeine_mg_before_bed', 'alcohol_units_before_bed']

for kolom in kolom_cek:
    tidak_standar = df[kolom].notna() & ~df[kolom].astype(str).str.fullmatch(r'-?\d+(\.\d+)?')
    print(kolom, '->', df.loc[tidak_standar, kolom].tolist())

age -> ['dua puluh', '29 thn']
sleep_duration_hrs -> ['delapan', '6,5', '7.2 jam']
steps_that_day -> ['8,524', '9.2k']
stress_score -> ['5,6']
heart_rate_resting_bpm -> ['68 bpm']
screen_time_before_bed_mins -> ['1.5 jam', 'N/A']
caffeine_mg_before_bed -> ['80mg']
alcohol_units_before_bed -> ['-']


In [240]:
# Angka berbahasa Indonesia sederhana angka ('delapan'8, 'dua puluh'20)
SATUAN = {'nol': 0, 'satu': 1, 'dua': 2, 'tiga': 3, 'empat': 4, 'lima': 5,
          'enam': 6, 'tujuh': 7, 'delapan': 8, 'sembilan': 9}

def kata_ke_angka(teks):
    total, cur = 0, 0
    for t in teks.split():
        if t in SATUAN:      cur += SATUAN[t]
        elif t == 'puluh':   total += max(cur, 1) * 10; cur = 0
        elif t == 'belas':   total += 10 + cur;         cur = 0
        elif t == 'sepuluh': total += 10
        elif t == 'sebelas': total += 11
        else:                return np.nan
    return float(total + cur)

In [241]:
# Parameter:
#   ribuan     : True jika koma atau titik di kolom itu adalah pemisah ribuan (kolom steps)
#   faktor_jam : pengali jika nilai ditulis 'jam' (kolom menit 60, kolom jam 1)
KOSONG = {'', 'n/a', 'na', '-', 'none', 'null', 'nan'}

def ke_angka(x, ribuan=False, faktor_jam=1):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    if s in KOSONG:
        return np.nan

    m = re.fullmatch(r'(-?[\d.,]+)\s*(k)?\s*([a-z]*)', s)
    if not m:                                   # bukan angka biasa coba sebagai kata
        return kata_ke_angka(s)

    angka, k, satuan = m.groups()
    if k:
        return float(angka.replace(',', '.')) * 1000
    if ribuan:
        angka = angka.replace(',', '')
        if re.fullmatch(r'\d{1,3}(\.\d{3})+', angka):
            angka = angka.replace('.', '')
    else:
        angka = angka.replace(',', '.')
    try:
        nilai = float(angka)
    except ValueError:
        return np.nan
    if satuan == 'jam':
        nilai *= faktor_jam
    return nilai

In [242]:
# Kolom numerik beserta opsi khususnya
kolom_numerik = {
    'age': {},
    'sleep_duration_hrs': {},
    'sleep_quality_score': {},
    'steps_that_day': {'ribuan': True},
    'stress_score': {},
    'heart_rate_resting_bpm': {},
    'screen_time_before_bed_mins': {'faktor_jam': 60},
    'caffeine_mg_before_bed': {},
    'alcohol_units_before_bed': {},
    'nap_duration_mins': {},
}

# Simpan nilai sebelum dibersihkan (untuk perbandingan)
sebelum = df[list(kolom_numerik)].copy()

# Terapkan fungsi pada setiap kolom
for kolom, opsi in kolom_numerik.items():
    df[kolom] = df[kolom].apply(lambda v: ke_angka(v, **opsi))

# Cek tipe data sekarang
df[list(kolom_numerik)].dtypes

,0
age,float64
sleep_duration_hrs,float64
sleep_quality_score,float64
steps_that_day,float64
stress_score,float64
heart_rate_resting_bpm,float64
screen_time_before_bed_mins,float64
caffeine_mg_before_bed,float64
alcohol_units_before_bed,float64
nap_duration_mins,float64


In [243]:
# Tampilkan nilai yang tadinya tidak standar: sebelum dan sesudah dibersihkan
for kolom in kolom_numerik:
    tidak_standar = sebelum[kolom].notna() & ~sebelum[kolom].astype(str).str.fullmatch(r'-?\d+(\.\d+)?')
    if tidak_standar.any():
        print('Kolom:', kolom)
        display(pd.DataFrame({'sebelum': sebelum.loc[tidak_standar, kolom],
                              'sesudah': df.loc[tidak_standar, kolom]}))

Kolom: age


,sebelum,sesudah
7,dua puluh,20.0
10,29 thn,29.0


Kolom: sleep_duration_hrs


,sebelum,sesudah
4,delapan,8.0
35,"6,5",6.5
40,7.2 jam,7.2


Kolom: steps_that_day


...

,sebelum,sesudah
0,1.5 jam,90.0
24,N/A,NaN


Kolom: caffeine_mg_before_bed


,sebelum,sesudah
16,80mg,80.0


Kolom: alcohol_units_before_bed


,sebelum,sesudah
27,-,NaN


# 8. Data Cleansing-Penanganan nilai tidak logis (outlier)


In [244]:
# Rentang nilai yang masuk akal untuk setiap kolom (batas bawah, batas atas)
rentang_valid = {
    'age': (10, 100),
    'sleep_duration_hrs': (1, 14),
    'sleep_quality_score': (0, 10),
    'steps_that_day': (0, 60000),
    'stress_score': (0, 10),
    'heart_rate_resting_bpm': (30, 150),
    'screen_time_before_bed_mins': (0, 600),
    'caffeine_mg_before_bed': (0, 1000),
    'alcohol_units_before_bed': (0, 20),
    'nap_duration_mins': (0, 300),
}

In [245]:
# Cari nilai di luar rentang, catat, lalu jadikan NaN
daftar_outlier = []

for kolom, (batas_bawah, batas_atas) in rentang_valid.items():
    # True jika nilai ada, tetapi di luar rentang
    di_luar = df[kolom].notna() & ~df[kolom].between(batas_bawah, batas_atas)

    # Catat setiap outlier untuk ditampilkan
    for idx in df.index[di_luar]:
        daftar_outlier.append({
            'no': df.loc[idx, 'no'],
            'kolom': kolom,
            'nilai': df.loc[idx, kolom],
            'rentang_valid': f'{batas_bawah} - {batas_atas}'
        })

    # Jadikan NaN
    df.loc[di_luar, kolom] = np.nan

# Tampilkan daftar outlier
pd.DataFrame(daftar_outlier)

,no,kolom,nilai,rentang_valid
0,14,age,-5.0,10 - 100
1,50,age,200.0,10 - 100
2,3,sleep_duration_hrs,-3.0,1 - 14
3,48,sleep_duration_hrs,25.0,1 - 14
4,28,sleep_quality_score,15.0,0 - 10
5,28,sleep_quality_score,15.0,0 - 10
6,38,sleep_quality_score,-1.0,0 - 10
7,6,steps_that_day,250000.0,0 - 60000
8,35,stress_score,99.0,0 - 10
9,7,heart_rate_resting_bpm,5.0,30 - 150


In [246]:
# Jumlah NaN per kolom setelah outlier dijadikan NaN
print(df[list(kolom_numerik)].isna().sum())

age                            3
sleep_duration_hrs             3
sleep_quality_score            4
steps_that_day                 2
stress_score                   2
heart_rate_resting_bpm         3
screen_time_before_bed_mins    1
caffeine_mg_before_bed         1
alcohol_units_before_bed       2
nap_duration_mins              2
dtype: int64


# 9. Data Cleansing-Deduplikasi Data

In [247]:
print('Jumlah baris sebelum deduplikasi :', len(df))
print('Duplikat berdasarkan no          :', df.duplicated(subset='no').sum())
print('Duplikat seluruh kolom           :', df.duplicated().sum())
print()

# Tampilkan semua baris yang nomornya kembar, urut berdasarkan no
df[df.duplicated(subset='no', keep=False)].sort_values('no')

Jumlah baris sebelum deduplikasi : 54
Duplikat berdasarkan no          : 4
Duplikat seluruh kolom           : 4



,no,age,gender,sleep_duration_hrs,sleep_quality_score,exercise_day,steps_that_day,stress_score,heart_rate_resting_bpm,screen_time_before_bed_mins,caffeine_mg_before_bed,alcohol_units_before_bed,nap_duration_mins
1,28,34.0,Female,5.36,NaN,1,12520.0,7.9,57.0,38.0,0.0,0.0,0.0
14,28,34.0,Female,5.36,NaN,1,12520.0,7.9,57.0,38.0,0.0,0.0,0.0
23,36,45.0,Female,6.44,5.2,1,6350.0,5.1,78.0,58.0,80.0,0.0,0.0
42,36,45.0,Female,6.44,5.2,1,6350.0,5.1,78.0,58.0,80.0,0.0,0.0
39,39,40.0,Female,7.18,4.6,0,2704.0,4.7,66.0,34.0,0.0,2.0,0.0
38,39,40.0,Female,7.18,4.6,0,2704.0,4.7,66.0,34.0,0.0,2.0,0.0
12,41,22.0,Male,4.05,2.3,0,4477.0,5.8,64.0,133.0,0.0,2.0,17.0
37,41,22.0,Male,4.05,2.3,0,4477.0,5.8,64.0,133.0,0.0,2.0,17.0


In [248]:
# Hapus duplikat berdasarkan no (pertahankan kemunculan pertama), urutkan, reset index
df = (df.drop_duplicates(subset='no', keep='first')
        .sort_values('no')
        .reset_index(drop=True))

print('Jumlah baris setelah deduplikasi :', len(df))
print('Nomor unik semua                 :', df['no'].is_unique)

Jumlah baris setelah deduplikasi : 50
Nomor unik semua                 : True


In [249]:
# Pastikan tidak ada duplikat tersisa dan nomornya lengkap 1 sampai 50
print('Duplikat tersisa        :', df.duplicated().sum())
print('Nomor lengkap 1 sampai 50:', df['no'].tolist() == list(range(1, 51)))

df.head(10)

Duplikat tersisa        : 0
Nomor lengkap 1 sampai 50: True


,no,age,gender,sleep_duration_hrs,sleep_quality_score,exercise_day,steps_that_day,stress_score,heart_rate_resting_bpm,screen_time_before_bed_mins,caffeine_mg_before_bed,alcohol_units_before_bed,nap_duration_mins
0,1,29.0,Female,6.50,6.6,0,6592.0,4.4,63.0,32.0,0.0,0.0,0.0
1,2,55.0,Female,8.32,6.9,1,10111.0,4.0,52.0,33.0,0.0,0.0,8.0
2,3,42.0,Male,NaN,1.0,1,9222.0,7.8,72.0,89.0,0.0,2.0,28.0
3,4,37.0,Male,6.79,6.4,1,9190.0,4.9,71.0,52.0,0.0,1.0,40.0
4,5,23.0,Male,5.02,3.2,0,4273.0,NaN,71.0,72.0,40.0,0.0,0.0
5,6,23.0,Female,8.16,5.7,0,NaN,5.5,71.0,72.0,150.0,2.0,0.0
6,7,20.0,Female,7.27,5.1,1,10786.0,5.0,NaN,148.0,0.0,3.0,0.0
7,8,48.0,Female,8.04,6.5,1,6194.0,5.5,60.0,59.0,150.0,0.0,46.0
8,9,37.0,Male,6.04,5.4,0,6083.0,6.0,NaN,40.0,400.0,0.0,44.0
9,10,41.0,Female,6.22,4.9,0,5207.0,5.7,80.0,88.0,0.0,0.0,NaN


# 10.  Data Cleansing-Penanganan Missing Value

In [250]:
print('Missing value sebelum ditangani:')
print(df.isna().sum()[lambda s: s > 0])
print()
print('Total sel kosong:', int(df.isna().sum().sum()))

Missing value sebelum ditangani:
age                            3
gender                         1
sleep_duration_hrs             3
sleep_quality_score            3
steps_that_day                 2
stress_score                   2
heart_rate_resting_bpm         3
screen_time_before_bed_mins    1
caffeine_mg_before_bed         1
alcohol_units_before_bed       2
nap_duration_mins              2
dtype: int64

Total sel kosong: 23


In [251]:
# Isi kolom numerik dengan median
for kolom in kolom_numerik:
    df[kolom] = df[kolom].fillna(df[kolom].median())

# Isi kolom kategori dengan modus (nilai yang paling sering muncul)
for kolom in ['gender', 'exercise_day']:
    df[kolom] = df[kolom].fillna(df[kolom].mode()[0])

print('Total sel kosong sekarang:', int(df.isna().sum().sum()))

Total sel kosong sekarang: 0


In [252]:
# Kolom yang seharusnya bilangan bulat
kolom_int = ['age', 'steps_that_day', 'heart_rate_resting_bpm', 'screen_time_before_bed_mins',
             'caffeine_mg_before_bed', 'alcohol_units_before_bed', 'nap_duration_mins']

df[kolom_int] = df[kolom_int].round().astype(int)
df['exercise_day'] = df['exercise_day'].astype(int)
df['gender'] = df['gender'].astype(str)

# Cek tipe data akhir
df.dtypes

,0
no,int64
age,int64
gender,object
sleep_duration_hrs,float64
sleep_quality_score,float64
exercise_day,int64
steps_that_day,int64
stress_score,float64
heart_rate_resting_bpm,int64
screen_time_before_bed_mins,int64


# 11. Data Enrichment

In [253]:
# Kelompok usia
df['kelompok_usia'] = pd.cut(df['age'], bins=[0, 24, 34, 49, 120],
                             labels=['Remaja-dewasa awal (<=24)', 'Dewasa muda (25-34)',
                                     'Dewasa (35-49)', 'Paruh baya (>=50)'])

# Kategori durasi tidur
df['kategori_durasi_tidur'] = pd.cut(df['sleep_duration_hrs'], bins=[0, 6, 9, 24],
                                     labels=['Kurang (<6 jam)', 'Cukup (6-9 jam)', 'Berlebih (>9 jam)'],
                                     right=False)

# Kategori kualitas tidur
df['kategori_kualitas_tidur'] = pd.cut(df['sleep_quality_score'], bins=[-0.01, 4, 7, 10],
                                       labels=['Rendah', 'Sedang', 'Tinggi'])

# Kategori tingkat stres
df['kategori_stres'] = pd.cut(df['stress_score'], bins=[-0.01, 3.99, 6.99, 10],
                              labels=['Rendah', 'Sedang', 'Tinggi'])

In [254]:
# Penanda konsumsi kafein dan alkohol sebelum tidur
df['konsumsi_kafein'] = np.where(df['caffeine_mg_before_bed'] > 0, 'Ya', 'Tidak')
df['konsumsi_alkohol'] = np.where(df['alcohol_units_before_bed'] > 0, 'Ya', 'Tidak')

# Total waktu tidur sehari (tidur malam + tidur siang) dalam jam
df['total_tidur_sehari_hrs'] = (df['sleep_duration_hrs'] + df['nap_duration_mins'] / 60).round(2)

In [255]:
df[['no', 'age', 'kelompok_usia', 'sleep_duration_hrs', 'kategori_durasi_tidur',
    'stress_score', 'kategori_stres', 'total_tidur_sehari_hrs']].head(10)

,no,age,kelompok_usia,sleep_duration_hrs,kategori_durasi_tidur,stress_score,kategori_stres,total_tidur_sehari_hrs
0,1,29,Dewasa muda (25-34),6.50,Cukup (6-9 jam),4.40,Sedang,6.50
1,2,55,Paruh baya (>=50),8.32,Cukup (6-9 jam),4.00,Sedang,8.45
2,3,42,Dewasa (35-49),6.57,Cukup (6-9 jam),7.80,Tinggi,7.04
3,4,37,Dewasa (35-49),6.79,Cukup (6-9 jam),4.90,Sedang,7.46
4,5,23,Remaja-dewasa awal (<=24),5.02,Kurang (<6 jam),5.55,Sedang,5.02
5,6,23,Remaja-dewasa awal (<=24),8.16,Cukup (6-9 jam),5.50,Sedang,8.16
6,7,20,Remaja-dewasa awal (<=24),7.27,Cukup (6-9 jam),5.00,Sedang,7.27
7,8,48,Dewasa (35-49),8.04,Cukup (6-9 jam),5.50,Sedang,8.81
8,9,37,Dewasa (35-49),6.04,Cukup (6-9 jam),6.00,Sedang,6.77
9,10,41,Dewasa (35-49),6.22,Cukup (6-9 jam),5.70,Sedang,6.22


In [256]:
# Jumlah responden per kategori
for kolom in ['kelompok_usia', 'kategori_durasi_tidur', 'kategori_kualitas_tidur',
              'kategori_stres', 'konsumsi_kafein', 'konsumsi_alkohol']:
    print(df[kolom].value_counts())
    print()

kelompok_usia
Dewasa muda (25-34)          18
Remaja-dewasa awal (<=24)    15
Dewasa (35-49)               12
Paruh baya (>=50)             5
Name: count, dtype: int64

kategori_durasi_tidur
Cukup (6-9 jam)      32
Kurang (<6 jam)      17
Berlebih (>9 jam)     1
Name: count, dtype: int64

kategori_kualitas_tidur
Sedang    35
Rendah    11
Tinggi     4
Name: count, dtype: int64

kategori_stres
Sedang    36
Tinggi     9
Rendah     5
Name: count, dtype: int64

konsumsi_kafein
Tidak    32
Ya       18
Name: count, dtype: int64

konsumsi_alkohol
Tidak    36
Ya       14
Name: count, dtype: int64



# 12. Validasi akhir dan simpan hasil

In [257]:
# Urutan kolom: identitas, data asli, lalu kolom hasil enrichment
urutan = ['no', 'age', 'gender', 'sleep_duration_hrs', 'sleep_quality_score',
          'exercise_day', 'steps_that_day', 'stress_score', 'heart_rate_resting_bpm',
          'screen_time_before_bed_mins', 'caffeine_mg_before_bed', 'alcohol_units_before_bed',
          'nap_duration_mins', 'kelompok_usia', 'kategori_durasi_tidur', 'kategori_kualitas_tidur',
          'kategori_stres', 'konsumsi_kafein', 'konsumsi_alkohol', 'total_tidur_sehari_hrs']

df = df[urutan]
df.shape

(50, 20)

In [258]:
# Ringkasan validasi: semua harus sesuai target
print('Jumlah baris         :', len(df))
print('Missing value        :', int(df.isna().sum().sum()))
print('Duplikat             :', int(df.duplicated().sum()))
print('Nomor unik 1 - 50    :', df['no'].tolist() == list(range(1, 51)))
print('Nilai gender         :', sorted(df['gender'].unique()))
print('Nilai exercise_day   :', sorted(df['exercise_day'].unique()))
print('Semua dalam rentang  :', all(df[k].between(lo, hi).all() for k, (lo, hi) in rentang_valid.items()))

Jumlah baris         : 50
Missing value        : 0
Duplikat             : 0
Nomor unik 1 - 50    : True
Nilai gender         : ['Female', 'Male']
Nilai exercise_day   : [np.int64(0), np.int64(1)]
Semua dalam rentang  : True


In [259]:
# Statistik deskriptif data bersih
df.describe().T

,count,mean,std,min,25%,50%,75%,max
no,50.0,25.5000,14.577380,1.00,13.2500,25.500,37.7500,50.00
age,50.0,32.2400,10.769496,18.00,24.0000,29.000,37.0000,58.00
sleep_duration_hrs,50.0,6.5968,1.255558,3.77,5.7725,6.570,7.5675,9.98
sleep_quality_score,50.0,4.9400,1.556946,1.00,4.3250,5.100,6.0500,7.40
exercise_day,50.0,0.5400,0.503457,0.00,0.0000,1.000,1.0000,1.00
steps_that_day,50.0,7657.9200,2893.826341,987.00,6110.7500,7835.000,9337.5000,13960.00
stress_score,50.0,5.6180,1.437322,2.20,4.8250,5.550,6.7000,8.80
heart_rate_resting_bpm,50.0,68.4200,7.236782,52.00,63.0000,70.000,72.7500,86.00
screen_time_before_bed_mins,50.0,68.6600,43.347485,13.00,35.7500,54.000,91.7500,180.00
caffeine_mg_before_bed,50.0,51.4000,95.810400,0.00,0.0000,0.000,80.0000,400.00


In [260]:
# Simpan hasil ke folder yang sama dengan file kotor
output_path = '/content/drive/MyDrive/Colab Notebooks/DATMIN/sleep_health_data_bersih.csv'
df.to_csv(output_path, index=False)
print('Tersimpan di:', output_path)

# Tampilkan data bersih
df.head(10)

Tersimpan di: /content/drive/MyDrive/Colab Notebooks/DATMIN/sleep_health_data_bersih.csv


,no,age,gender,sleep_duration_hrs,sleep_quality_score,exercise_day,steps_that_day,stress_score,heart_rate_resting_bpm,screen_time_before_bed_mins,caffeine_mg_before_bed,alcohol_units_before_bed,nap_duration_mins,kelompok_usia,kategori_durasi_tidur,kategori_kualitas_tidur,kategori_stres,konsumsi_kafein,konsumsi_alkohol,total_tidur_sehari_hrs
0,1,29,Female,6.50,6.6,0,6592,4.40,63,32,0,0,0,Dewasa muda (25-34),Cukup (6-9 jam),Sedang,Sedang,Tidak,Tidak,6.50
1,2,55,Female,8.32,6.9,1,10111,4.00,52,33,0,0,8,Paruh baya (>=50),Cukup (6-9 jam),Sedang,Sedang,Tidak,Tidak,8.45
2,3,42,Male,6.57,1.0,1,9222,7.80,72,89,0,2,28,Dewasa (35-49),Cukup (6-9 jam),Rendah,Tinggi,Tidak,Ya,7.04
3,4,37,Male,6.79,6.4,1,9190,4.90,71,52,0,1,40,Dewasa (35-49),Cukup (6-9 jam),Sedang,Sedang,Tidak,Ya,7.46
4,5,23,Male,5.02,3.2,0,4273,5.55,71,72,40,0,0,Remaja-dewasa awal (<=24),Kurang (<6 jam),Rendah,Sedang,Ya,Tidak,5.02
5,6,23,Female,8.16,5.7,0,7835,5.50,71,72,150,2,0,Remaja-dewasa awal (<=24),Cukup (6-9 jam),Sedang,Sedang,Ya,Ya,8.16
6,7,20,Female,7.27,5.1,1,10786,5.00,70,148,0,3,0,Remaja-dewasa awal (<=24),Cukup (6-9 jam),Sedang,Sedang,Tidak,Ya,7.27
7,8,48,Female,8.04,6.5,1,6194,5.50,60,59,150,0,46,Dewasa (35-49),Cukup (6-9 jam),Sedang,Sedang,Ya,Tidak,8.81
8,9,37,Male,6.04,5.4,0,6083,6.00,70,40,400,0,44,Dewasa (35-49),Cukup (6-9 jam),Sedang,Sedang,Ya,Tidak,6.77
9,10,41,Female,6.22,4.9,0,5207,5.70,80,88,0,0,0,Dewasa (35-49),Cukup (6-9 jam),Sedang,Sedang,Tidak,Tidak,6.22


In [261]:
import os
print(os.path.exists(output_path))

True
